In [5]:
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
import torch
import lightning as pl
import wandb
from torchmetrics.classification import Accuracy
from lightning.pytorch.loggers import WandbLogger

from configs import configs

In [6]:
x_test = pd.read_csv("data/processed_data/x_test.csv").astype(float)
y_test = pd.read_csv("data/processed_data/y_test.csv").astype(float)

In [3]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.tensor(self.x.iloc[idx].values, dtype=torch.float32), torch.tensor(self.y.iloc[idx].values, dtype=torch.float32)

In [4]:
import pandas as pd
import numpy as np
import torch
import lightning as pl
from torchmetrics.classification import Accuracy, F1Score

from configs import configs

class HeartDataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.tensor(self.x.iloc[idx].values, dtype=torch.float32), torch.tensor(self.y.iloc[idx].values, dtype=torch.float32)

class CustomModel(pl.LightningModule):
    def __init__(self, lr):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(19, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid()
        )
        self.train_accuracy = Accuracy(task="binary")
        self.val_accuracy = Accuracy(task="binary")
        self.train_f1 = F1Score(num_classes=2, task="binary")
        self.val_f1 = F1Score(num_classes=2, task="binary")
        self.loss_fn = torch.nn.BCELoss()
        self.lr = lr

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.train_accuracy.update(y_hat, y)
        self.train_f1.update(y_hat, y)
        self.log("train/loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.val_accuracy.update(y_hat, y)
        self.val_f1.update(y_hat, y)
        self.log("val/loss", loss)
        return loss

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x, y = batch
        return self(x)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)
    
    def on_train_epoch_end(self):
        self.log("train_acc_epoch", self.train_accuracy.compute())
        self.log("train_f1_epoch", self.train_f1.compute())
    
    def on_validation_epoch_end(self):
        self.log("val_acc_epoch", self.val_accuracy.compute())
        self.log("val_f1_epoch", self.val_f1.compute())

x_test = pd.read_csv("data/processed_data/x_test.csv").astype(float)
y_test = pd.read_csv("data/processed_data/y_test.csv").astype(float)

test_dataset = HeartDataset(x_test, y_test)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=configs["batch_size"], shuffle=False)

model = CustomModel.load_from_checkpoint("models\epoch=8-step=162-v1.ckpt", lr=configs["lr"])

trainer = pl.Trainer(accelerator="gpu", devices=[0])
trainer.validate(model, test_dataloader)
test_predictions = trainer.predict(model, test_dataloader)


<>:78: SyntaxWarning: invalid escape sequence '\e'
<>:78: SyntaxWarning: invalid escape sequence '\e'
C:\Users\avhrs\AppData\Local\Temp\ipykernel_20424\2203647669.py:78: SyntaxWarning: invalid escape sequence '\e'
  model = CustomModel.load_from_checkpoint("models\epoch=8-step=162-v1.ckpt", lr=configs["lr"])
C:\Users\avhrs\AppData\Local\Temp\ipykernel_20424\2203647669.py:78: SyntaxWarning: invalid escape sequence '\e'
  model = CustomModel.load_from_checkpoint("models\epoch=8-step=162-v1.ckpt", lr=configs["lr"])


FileNotFoundError: [Errno 2] No such file or directory: 'c:/Users/avhrs/Developer/python-learn/models/epoch=8-step=162-v1.ckpt'